In [1]:
import os
import pandas as pd
import geopandas as gpd
import fiona
fiona.drvsupport.supported_drivers['KML'] = 'rw' # enable KML support which is disabled by default
import gc
import dask.dataframe as dd

pd.set_option('display.max_rows', 500)

from utilities import walk_through_files, auto_reduce_mem

In [2]:
FP_SUBZONES_SHP = os.path.join('zip://' '..', 'raw-files', 'master-plan-2014-subzone-boundary-no-sea', 'master-plan-2014-subzone-boundary-no-sea-kml.zip')
FP_PARKS = os.path.join('..', 'raw-files', 'poi', 'parks', 'parks-geojson.geojson')
FP_LIB = os.path.join('..', 'raw-files', 'poi', 'libraries', 'libraries-geojson.geojson')
FP_TRANSIT = os.path.join('..', 'raw-files', 'poi', 'lta-mrt-station-exit', 'lta-mrt-station-exit-geojson.geojson')
FP_PRESCH = os.path.join('..', 'raw-files', 'poi', 'pre-schools-location', 'pre-schools-location-geojson.geojson')
FP_SPORT = os.path.join('..', 'raw-files', 'poi', 'sportsg-sport-facilities', 'sportsg-sport-facilities-geojson.geojson')
FP_MART = os.path.join('..', 'raw-files', 'poi', 'supermarkets', 'supermarkets-geojson.geojson')
FP_SCHZONE = os.path.join('..', 'raw-files', 'poi', 'lta-school-zone', 'lta-school-zone-kml.kml')
FP_SILVER = os.path.join('..', 'raw-files', 'poi', 'lta-silver-zone', 'lta-silver-zone-kml.kml')
FP_TOURIST = os.path.join('zip://', '..', 'raw-files', 'poi', 'tourist-attractions', 'tourist-attractions-kml.zip')
FP_LRT = os.path.join('..', 'raw-files', 'poi', 'singapore-postal-codes-master', 'lrt_stations.json')
FP_MRT = os.path.join('..', 'raw-files', 'poi', 'singapore-postal-codes-master', 'mrt_stations.json')
FP_BUILDINGS = os.path.join('..', 'raw-files', 'poi', 'singapore-postal-codes-master', 'buildings.json')

FP_REG_BUSINESS = os.path.join('..', '..', 'wgw-lf', 'acra-information-on-corporate-entities')

FP_XWALK_ZIP_SUBZONE = os.path.join('intermediate-files', 'fixed_postal_subzone_xwalk.csv')

SAVEPATH = os.path.join('intermediate-files', 'subzones-poi-count.csv')

In [3]:
xwalk = (pd.read_csv(FP_XWALK_ZIP_SUBZONE)
         .rename(columns={'POSTAL': 'postcode', 'subzone_name': 'subzone'})
         .assign(postcode=lambda df: df['postcode'].astype('str').str.zfill(6))
        )

### Data.gov.sg resources

In [4]:
subzones = (gpd.read_file(FP_SUBZONES_SHP)
            .rename(columns={'Name':'subzone'})
            .drop('Description', axis=1)
           )

In [5]:
# Parks
parks = (gpd.read_file(FP_PARKS) 
         .rename(columns={'Name':'park'})
         .drop('Description', axis=1)
        )

assert parks.crs==subzones.crs

parks = (gpd.sjoin(parks, subzones, how='left', op='within')
         .groupby('subzone').size().reset_index()
         .rename(columns={0:'parks'})
        )

In [6]:
# Libraries
lib = (gpd.read_file(FP_LIB)
       .rename(columns={'Name':'libraries'})
       .drop('Description', axis=1)
      )

assert lib.crs == subzones.crs

lib = (gpd.sjoin(lib, subzones, how='left', op='within')
         .groupby('subzone').size().reset_index()
         .rename(columns={0:'libraries'})
        )

In [7]:
# LRT & MRT stations
transit = (gpd.read_file(FP_TRANSIT)
           .rename(columns={'Name':'transit'})
           .drop('Description', axis=1)
          )

assert transit.crs == subzones.crs

transit = (gpd.sjoin(transit, subzones, how='left', op='within')
           .groupby('subzone').size().reset_index()
           .rename(columns={0:'transit'})
          )

In [8]:
# Preschools
presch = (gpd.read_file(FP_PRESCH)
          .rename(columns={'Name':'preschools'})
          .drop('Description', axis=1)
         )

assert presch.crs == subzones.crs

presch = (gpd.sjoin(presch, subzones, how='left', op='within')
         .groupby('subzone').size().reset_index()
         .rename(columns={0:'preschools'})
        )

In [9]:
# Sport facilities
sport = (gpd.read_file(FP_SPORT)
         .rename(columns={'Name':'sport_facilities'})
         .drop('Description', axis=1)
      )

assert sport.crs == subzones.crs

sport = (gpd.sjoin(sport, subzones, how='left', op='within')
         .groupby('subzone').size().reset_index()
         .rename(columns={0:'sport_facilities'})
        )

In [10]:
# Supermarkets
marts = (gpd.read_file(FP_MART)
         .rename(columns={'Name':'supermarkets'})
         .drop('Description', axis=1)
        )

assert marts.crs == subzones.crs

marts = (gpd.sjoin(marts, subzones, how='left', op='within')
         .groupby('subzone').size().reset_index()
         .rename(columns={0:'supermarkets'})
        )

In [11]:
# Supermarkets
marts = (gpd.read_file(FP_MART)
         .rename(columns={'Name':'supermarkets'})
         .drop('Description', axis=1)
        )

assert marts.crs == subzones.crs

marts = (gpd.sjoin(marts, subzones, how='left', op='within')
         .groupby('subzone').size().reset_index()
         .rename(columns={0:'supermarkets'})
        )

In [12]:
# LTA school zones
schzone = (gpd.read_file(FP_SCHZONE)
         .rename(columns={'Name':'schzone'})
         .drop('Description', axis=1)
        )

assert schzone.crs == subzones.crs

schzone = (gpd.sjoin(schzone, subzones, how='left', op='intersects')
         .groupby('subzone').size().reset_index()
         .rename(columns={0:'schzones'})
        )

In [13]:
# LTA silver zones
silver = (gpd.read_file(FP_SILVER)
         .rename(columns={'Name':'silverzones'})
         .drop('Description', axis=1)
        )

assert silver.crs == subzones.crs

silver = (gpd.sjoin(silver, subzones, how='left', op='intersects')
         .groupby('subzone').size().reset_index()
         .rename(columns={0:'silverzones'})
        )

In [14]:
# Tourist attractions
tour = (gpd.read_file(FP_TOURIST)
        .rename(columns={'Name':'tourist_attraction'})
        .drop('Description', axis=1)
       )

assert tour.crs == subzones.crs

tour = (gpd.sjoin(tour, subzones, how='left', op='intersects')
        .groupby('subzone').size().reset_index()
        .rename(columns={0:'tour'})
       )

### Open-source resources

#### Schools

In [15]:
poi = (pd.read_json(FP_BUILDINGS)
               .rename(columns={'BUILDING': 'poi', 'POSTAL': 'postcode'})
               .filter(['poi', 'postcode'])
               .assign(poi=lambda df: df['poi'].str.lower())
      )

schools = poi[(poi['poi'].str.contains('primary school')) | (poi['poi'].str.contains('secondary school'))]

del poi
gc.collect()

# Merge schools with postcode to subzone crosswalk
schools = (schools.merge(xwalk, how='left', on='postcode')
           .drop('postcode', axis=1)
           .groupby('subzone').size().reset_index()
           .rename(columns={0: 'schools'})
          )

#### Transit

In [16]:
lrts = (pd.read_json(FP_LRT)
        .rename(columns={'Possible Locations': 'geopayload'})
        .explode('geopayload')
        .query('geopayload==geopayload')
        .assign(postcode=lambda df: [js['POSTAL'] for js in df['geopayload']])
        .drop_duplicates('Station')
        .merge(xwalk, how='left', on='postcode')
        .groupby('subzone').size().reset_index()
        .rename(columns={0: 'lrts'})       
       )

In [17]:
mrts = (pd.read_json(FP_MRT)
        .rename(columns={'Possible Locations': 'geopayload'})
        .explode('geopayload')
        .query('geopayload==geopayload')
        .assign(postcode=lambda df: [js['POSTAL'] for js in df['geopayload']])
        .drop_duplicates('Station')
        .merge(xwalk, how='left', on='postcode')
        .groupby('subzone').size().reset_index()
        .rename(columns={0: 'mrts'})
       )

#### Distribution of industry groups
Resources on SSIC:

* https://stats.mom.gov.sg/Pages/Employment-Distribution-and-Employment-Level-by-Industry.aspx\
* https://www.singstat.gov.sg/standards/standards-and-classifications/ssic
* https://www.singstat.gov.sg/-/media/files/standards_and_classifications/industrial_classification/ssic2020report.pdf

3 Biggest "Industry groups:

1. Manufacturing 10
2. construction 41
3. Services:
* wholesale 46
* retail trade 47
* Transport & storage 49
* Land transport & Support svcs 49?

In [18]:
ddf = (dd.read_csv(list(walk_through_files(FP_REG_BUSINESS, 'csv', fullpath=True)))
       .assign(entity_status_description=lambda ddf: ddf['entity_status_description'].str.lower())
       .assign(entity_status_description=lambda ddf: ddf['entity_status_description'].str.replace('live company', 'live'))
       .query('entity_status_description=="live"')
       .rename(columns={'primary_ssic_code': 'ssic'})
       .pipe(lambda ddf: ddf[['postal_code', 'ssic']])
       .astype({'ssic': 'str'})
       .assign(ind=lambda ddf: ddf['ssic'].str.zfill(5).str.slice(stop=2))
       .groupby(['postal_code', 'ind']).size().reset_index()
       .rename(columns={0: 'count', 'postal_code': 'postcode'})
      ).compute()

manufact = ddf[ddf['ind']=='10']
construct = ddf[ddf['ind']=='41']
services = ddf[(ddf['ind']=='46') | (ddf['ind']=='47') | (ddf['ind']=='49')]

# del ddf
# gc.collect()

D:\ProgramData\Anaconda3\lib\site-packages\dask\core.py:121: DtypeWarning: Columns (6) have mixed types.Specify dtype option on import or set low_memory=False.
  return func(*(_execute_task(a, cache) for a in args))
D:\ProgramData\Anaconda3\lib\site-packages\dask\core.py:121: DtypeWarning: Columns (6) have mixed types.Specify dtype option on import or set low_memory=False.
  return func(*(_execute_task(a, cache) for a in args))


In [19]:
manufact = (manufact.merge(xwalk, how='left', on='postcode')
            .groupby('subzone')['count'].sum().reset_index()
            .rename(columns={'count':'manufacturing'})
           )


construct = (construct.merge(xwalk, how='left', on='postcode')
             .groupby('subzone')['count'].sum().reset_index()
             .rename(columns={'count':'construction'})
            )


services = (services.merge(xwalk, how='left', on='postcode')
            .groupby('subzone')['count'].sum().reset_index()            
            .rename(columns={'count':'services'})           
           )


In [23]:
df_list_toappend = [lib, marts, parks, presch, schzone, silver, sport, transit, tour, 
                    schools, mrts, lrts, manufact, construct, services]

# Merge
df = subzones[['subzone']]

for _pddf in df_list_toappend:
    df = df.merge(_pddf, how='outer', on='subzone')

for _pddf in df_list_toappend:
    del _pddf
    gc.collect()
    
df.fillna(0, inplace=True)

auto_reduce_mem(df, downcast_float='integer')

Memory usage of DataFrame is 0.0 (0.1) MB.
******************************
Column: subzone
dtype before: object
dtype after: category
******************************

******************************
Column: libraries
dtype before: float64
dtype after: int8
******************************

******************************
Column: supermarkets
dtype before: float64
dtype after: int8
******************************

******************************
Column: parks
dtype before: float64
dtype after: int8
******************************

******************************
Column: preschools
dtype before: float64
dtype after: int8
******************************

******************************
Column: schzones
dtype before: float64
dtype after: int8
******************************

******************************
Column: silverzones
dtype before: float64
dtype after: int8
******************************

******************************
Column: sport_facilities
dtype before: float64
dtype after: int8
************

In [24]:
df

,subzone,libraries,supermarkets,parks,preschools,schzones,silverzones,sport_facilities,transit,tour,schools,mrts,lrts,manufacturing,construction,services
0,PEOPLE'S PARK,0,0,0,0,0,0,0,5,0,0,2,0,4,1,172
1,BUKIT MERAH,0,1,0,3,0,0,0,0,1,0,0,0,15,88,974
2,CHINATOWN,0,3,3,10,0,0,0,3,8,0,1,0,46,126,2418
3,PHILLIP,0,0,0,1,0,0,0,0,0,0,0,0,1,2,137
4,RAFFLES PLACE,0,0,1,3,0,0,0,12,0,0,2,0,31,57,4582
5,CHINA SQUARE,1,3,0,0,0,0,0,3,2,0,2,0,25,47,1361
6,TIONG BAHRU,0,1,1,5,0,0,0,0,1,0,0,0,8,7,271
7,BAYFRONT SUBZONE,0,1,0,0,0,0,0,5,5,0,2,0,0,0,7
8,TIONG BAHRU STATION,0,3,0,3,1,0,0,2,0,1,1,0,0,6,180
9,CLIFFORD PIER,0,0,1,1,0,0,0,0,1,0,0,0,3,1,105


In [26]:
df.to_csv(SAVEPATH, index=False)